# Using and sharing models from `aiondemand`

`aiondemand` key features are:

* one-stop-shop to obtain AI models from simple string representations
* catalogue to search and share data about research projects and publications

This guide demonstrates string-indexing of models from popular machine learning libraries and platforms.

(NOTE: `aiondemand` is in development. Currently, only `scikit-learn`-like libraries are supported)

`aiod.get(id: str)` is the key entry point:

* `id` is a unique string describing a model
* return is the python object, or an exception informing the user of required dependencies

In [1]:
from aiod import get

#### Example: obtaining a `scikit-learn` classifier

In [2]:
clf = get("RandomForestClassifier")

this is the `scikit-learn` object for direct use:

In [3]:
clf

sklearn.ensemble._forest.RandomForestClassifier

works for classes as well as for instances:

In [4]:
get("RandomForestClassifier(n_estimators=42)")

RandomForestClassifier(n_estimators=42)

#### Example: complex ML pipeline

`aiondemand` also works for complex ML pipelines:

In [5]:
pipeline_spec = """
Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100))
])
"""

In [6]:
get(pipeline_spec)

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler()),
                ('classifier', RandomForestClassifier())])

#### What is this useful for?

* direct use for developers to avoid the hassle of looking up import paths
* treating AI algorithm specs as strings
    * directly print `scikit-learn` compatible algorithms to get the string!
    * convert strings to algorithm via `aiod.get`
* storing and sharing AI specs

## Indexed Libraries

AIoD indexes estimators from the following popular machine learning libraries and scikit-learn compatible packages (see [issue #57](https://github.com/aiondemand/aiondemand/issues/57)):

`scikit-learn` tabular estimators

- `scikit-learn`
- `catboost`
- `feature-engine`
- `lightgbm`
- `imbalanced-learn`
- `mlxtend`
- `scikit-lego`
- `xgboost`

Probabilistic supervised learning and survival modelling

- `skpro`

Time series

- `sktime`

All indexed estimators can be accessed uniformly through `aiod.get()`.

This supports composites with components across libraries!

## 1. Retrieving Models

### Getting Model Classes

Access any indexed model class using its name. AIoD automatically handles finding the estimator across indexed libraries:

In [7]:
import aiod

# Scikit-learn classifiers and transformers
RandomForestClassifier = aiod.get("RandomForestClassifier")
LGBMClassifier = aiod.get("LGBMClassifier")
CatBoostClassifier = aiod.get("CatBoostClassifier")

RuntimeError: class LGBMClassifier is required to build spec, but get('LGBMClassifier') failed

If a required library is not installed, AIoD will raise an error message:

```
ModuleNotFoundError: LGBMClassifier requires package 'lightgbm' to be present in the python environment, but 'lightgbm' was not found.
```

### Instantiating Models with Hyperparameters

Create model instances directly with specific hyperparameters:

In [8]:
import aiod

# Simple classifier with hyperparameters
rf = aiod.get("RandomForestClassifier(n_estimators=100)")
rf

RandomForestClassifier()

In [9]:
# Preprocessing pipeline transformations also work
imputer = get("SimpleImputer(strategy='mean')")
scaler = get("StandardScaler()")

### Linear Pipelines

Compose multi-step pipelines from string specifications:

In [ ]:
pipeline = get("""
Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100))
])
""")

print(type(pipeline))
# <class 'sklearn.pipeline.Pipeline'>

<class 'sklearn.pipeline.Pipeline'>


In [11]:
pipeline

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler()),
                ('classifier', RandomForestClassifier())])

### Complex composites: pipeline and tuning

`get` supports arbitrary compositions, e.g., pipelines and tuning:

In [12]:
import aiod

spec = """
pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=100))])
cv = KFold(n_splits=5, shuffle=True, random_state=42)

return GridSearchCV(
    estimator=pipe,
    param_grid=[{
        "classifier__max_depth": [5, 10],
        "classifier__min_samples_split": [2, 5],
    },
    ],
    cv=cv,
    )
"""

get(spec)

GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('imputer', SimpleImputer()),
                                       ('scaler', StandardScaler()),
                                       ('classifier',
                                        RandomForestClassifier())]),
             param_grid=[{'classifier__max_depth': [5, 10],
                          'classifier__min_samples_split': [2, 5]}])

# ROADMAP

* extending indexing scope, e.g., deep learning architectures
* cross-linkage to metadata catalogue
* datasets and experiments